In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("../output_csv")

input_file = folder / "residential_cleaned_preprocessed.csv"

df = pd.read_csv(input_file, low_memory=False)

df.head()

,ClosePrice,source_month,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,PostalCode_92563,PostalCode_92584,PostalCode_92592,PostalCode_92596,PostalCode_93065,PostalCode_93535,PostalCode_93536,PostalCode_93551,PostalCode_94513,PostalCode_Other
0,1800000.0,202505,1.448510,-0.753592,-0.020607,0.984604,1.229861,1.602918,0.306167,-0.475161,...,False,False,False,False,False,False,False,False,False,True
1,1200000.0,202505,-0.853308,-0.753592,-0.020792,-1.299200,-0.561218,-1.572704,-0.001975,-0.363308,...,False,False,False,False,False,False,False,False,False,True
2,2250000.0,202505,-0.880343,-0.753592,-0.020726,-0.755437,-1.456758,-0.514163,-0.310118,1.467452,...,False,False,False,False,False,False,False,False,False,True
3,1425000.0,202505,-0.360889,-0.753592,-0.020622,0.440841,-0.561218,-0.514163,-0.001975,-0.483640,...,False,False,False,False,False,False,False,False,False,True
4,660000.0,202505,-0.733583,-0.753592,-0.020800,0.368339,0.334321,-0.514163,-0.001975,-0.890424,...,False,False,False,False,False,False,False,False,False,True


In [2]:
target = 'ClosePrice'
# Train/test split by most recent month
df['source_month'] = df['source_month'].astype(float).astype(int).astype(str)

months = sorted(df['source_month'].unique())

test_month = months[-1]

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

target = "ClosePrice"

df["source_month"] = (
    df["source_month"]
    .astype(float)
    .astype(int)
    .astype(str)
)

months = sorted(df["source_month"].unique())
test_month = months[-1]

drop_cols = [target, "source_month"]

feature_cols = [
    col for col in df.columns
    if col not in drop_cols
]

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

model_results = []

for X_window in [3, 6, 9, 12]:

    train_months = months[-(X_window + 1):-1]

    train_df = df[
        df["source_month"].isin(train_months)
    ].copy()

    test_df = df[
        df["source_month"] == test_month
    ].copy()

    X_train = train_df[feature_cols]
    y_train = train_df[target]

    X_test = test_df[feature_cols]
    y_test = test_df[target]

    for model_name, estimator in models.items():

        # Fresh copy of each estimator for every training window
        model = clone(estimator)

        model.fit(X_train, y_train)

        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)

        model_results.append({
            "model": model_name,
            "training_window_months": X_window,
            "train_months": ", ".join(train_months),
            "test_month": test_month,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "feature_count": len(feature_cols),
            "train_r2": r2_score(
                y_train,
                train_preds
            ),
            "test_r2": r2_score(
                y_test,
                test_preds
            ),
            "mae": mean_absolute_error(
                y_test,
                test_preds
            ),
            "rmse": np.sqrt(
                mean_squared_error(
                    y_test,
                    test_preds
                )
            )
        })

comparison_df = pd.DataFrame(model_results)

comparison_df = comparison_df.sort_values(
    by="test_r2",
    ascending=False
).reset_index(drop=True)

comparison_df

In [ ]:
# Find the best Linear Regression baseline
best_baseline_row = (
    comparison_df[
        comparison_df["model"] == "Linear Regression"
    ]
    .sort_values(
        by="test_r2",
        ascending=False
    )
    .iloc[0]
)

best_baseline_r2 = best_baseline_row["test_r2"]
best_baseline_window = best_baseline_row["training_window_months"]

print("Best baseline window:", best_baseline_window)
print(f"Best baseline test R²: {best_baseline_r2:.4f}")

Best baseline window: 9
Best baseline test R²: -353792175596930.3750


In [ ]:
comparison_df["baseline_test_r2"] = best_baseline_r2

comparison_df["r2_change_from_baseline"] = (
    comparison_df["test_r2"]
    - comparison_df["baseline_test_r2"]
)

comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,model,training_window_months,train_months,test_month,lower_price_cutoff,upper_price_cutoff,train_rows,test_rows,train_r2,test_r2,mae,rmse,baseline_test_r2,r2_change_from_baseline
0,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,190000.0,8900000.0,128039,11863,0.982727,8.759030e-01,1.717792e+05,3.515082e+05,-3.537922e+14,3.537922e+14
1,Random Forest,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,188585.0,9000000.0,92982,11866,0.981367,8.671544e-01,1.778947e+05,3.655043e+05,-3.537922e+14,3.537922e+14
2,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,185000.0,9130625.0,58569,11871,0.979720,8.540990e-01,1.870421e+05,3.840166e+05,-3.537922e+14,3.537922e+14
3,Random Forest,3,"202602, 202603, 202604",202605,189072.0,9246560.0,31299,11866,0.978117,8.431746e-01,1.982061e+05,3.981239e+05,-3.537922e+14,3.537922e+14
4,Decision Tree,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,190000.0,8900000.0,128039,11863,0.999772,7.465204e-01,2.452290e+05,5.023730e+05,-3.537922e+14,3.537922e+14
5,Decision Tree,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,188585.0,9000000.0,92982,11866,0.999991,7.363041e-01,2.554043e+05,5.149565e+05,-3.537922e+14,3.537922e+14
6,Decision Tree,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,185000.0,9130625.0,58569,11871,0.999995,7.245473e-01,2.640347e+05,5.276482e+05,-3.537922e+14,3.537922e+14
7,Decision Tree,3,"202602, 202603, 202604",202605,189072.0,9246560.0,31299,11866,1.000000,6.864425e-01,2.777771e+05,5.629486e+05,-3.537922e+14,3.537922e+14
8,Linear Regression,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,188585.0,9000000.0,92982,11866,0.668662,-3.537922e+14,1.731576e+11,1.886223e+13,-3.537922e+14,0.000000e+00
9,Linear Regression,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,185000.0,9130625.0,58569,11871,0.671206,-6.367341e+14,2.328397e+11,2.536880e+13,-3.537922e+14,-2.829419e+14


In [ ]:
comparison_df["beat_baseline"] = (
    comparison_df["test_r2"] > best_baseline_r2
)

comparison_df[
    [
        "model",
        "training_window_months",
        "test_r2",
        "baseline_test_r2",
        "r2_change_from_baseline",
        "beat_baseline"
    ]
].sort_values(
    by="test_r2",
    ascending=False
)

,model,training_window_months,test_r2,baseline_test_r2,r2_change_from_baseline,beat_baseline
0,Random Forest,12,8.759030e-01,-3.537922e+14,3.537922e+14,True
1,Random Forest,9,8.671544e-01,-3.537922e+14,3.537922e+14,True
2,Random Forest,6,8.540990e-01,-3.537922e+14,3.537922e+14,True
3,Random Forest,3,8.431746e-01,-3.537922e+14,3.537922e+14,True
4,Decision Tree,12,7.465204e-01,-3.537922e+14,3.537922e+14,True
5,Decision Tree,9,7.363041e-01,-3.537922e+14,3.537922e+14,True
6,Decision Tree,6,7.245473e-01,-3.537922e+14,3.537922e+14,True
7,Decision Tree,3,6.864425e-01,-3.537922e+14,3.537922e+14,True
8,Linear Regression,9,-3.537922e+14,-3.537922e+14,0.000000e+00,False
9,Linear Regression,6,-6.367341e+14,-3.537922e+14,-2.829419e+14,False


In [ ]:
df_engineered = df.copy()

# Convert source month into year and month components
df_engineered["sale_year"] = (
    df_engineered["source_month"]
    .str[:4]
    .astype(int)
)

df_engineered["sale_month_number"] = (
    df_engineered["source_month"]
    .str[4:6]
    .astype(int)
)

df_engineered["sale_quarter"] = (
    (df_engineered["sale_month_number"] - 1) // 3 + 1
)

# Cyclical month features
df_engineered["month_sin"] = np.sin(
    2 * np.pi * df_engineered["sale_month_number"] / 12
)

df_engineered["month_cos"] = np.cos(
    2 * np.pi * df_engineered["sale_month_number"] / 12
)

In [ ]:
# property age features


if "YearBuilt" in df_engineered.columns:

    df_engineered["property_age"] = (
        df_engineered["sale_year"]
        - df_engineered["YearBuilt"]
    )

    # Remove impossible negative ages
    df_engineered.loc[
        df_engineered["property_age"] < 0,
        "property_age"
    ] = np.nan

    df_engineered["property_age_squared"] = (
        df_engineered["property_age"] ** 2
    )

    df_engineered["is_new_property"] = (
        df_engineered["property_age"] <= 5
    ).astype(int)

    df_engineered["is_old_property"] = (
        df_engineered["property_age"] >= 50
    ).astype(int)

In [ ]:
# bedroom and bathroom features 

if (
    "BedroomsTotal" in df_engineered.columns
    and "BathroomsTotalInteger" in df_engineered.columns
):

    bedrooms = df_engineered["BedroomsTotal"]
    bathrooms = df_engineered["BathroomsTotalInteger"]

    df_engineered["bed_bath_ratio"] = (
        bedrooms
        / bathrooms.replace(0, np.nan)
    )

    df_engineered["bath_bed_ratio"] = (
        bathrooms
        / bedrooms.replace(0, np.nan)
    )

    df_engineered["total_bed_bath"] = (
        bedrooms + bathrooms
    )

    df_engineered["bed_bath_difference"] = (
        bedrooms - bathrooms
    )

    df_engineered["more_bathrooms_than_bedrooms"] = (
        bathrooms > bedrooms
    ).astype(int)

In [ ]:
# living area features 

if "LivingArea" in df_engineered.columns:

    df_engineered["living_area_squared"] = (
        df_engineered["LivingArea"] ** 2
    )

    df_engineered["log_living_area"] = np.log1p(
        df_engineered["LivingArea"].clip(lower=0)
    )

    df_engineered["small_home_flag"] = (
        df_engineered["LivingArea"] < 1000
    ).astype(int)

    df_engineered["large_home_flag"] = (
        df_engineered["LivingArea"] > 3000
    ).astype(int)


if (
    "LivingArea" in df_engineered.columns
    and "BedroomsTotal" in df_engineered.columns
):

    df_engineered["living_area_per_bedroom"] = (
        df_engineered["LivingArea"]
        / df_engineered["BedroomsTotal"].replace(0, np.nan)
    )


if (
    "LivingArea" in df_engineered.columns
    and "BathroomsTotalInteger" in df_engineered.columns
):

    df_engineered["living_area_per_bathroom"] = (
        df_engineered["LivingArea"]
        / df_engineered[
            "BathroomsTotalInteger"
        ].replace(0, np.nan)
    )

In [ ]:
# lot size features

if "LotSizeSquareFeet" in df_engineered.columns:

    df_engineered["log_lot_size"] = np.log1p(
        df_engineered[
            "LotSizeSquareFeet"
        ].clip(lower=0)
    )

    df_engineered["large_lot_flag"] = (
        df_engineered["LotSizeSquareFeet"] > 10000
    ).astype(int)

    df_engineered["small_lot_flag"] = (
        df_engineered["LotSizeSquareFeet"] < 3000
    ).astype(int)



if (
    "LotSizeSquareFeet" in df_engineered.columns
    and "LivingArea" in df_engineered.columns
):

    df_engineered["lot_to_living_ratio"] = (
        df_engineered["LotSizeSquareFeet"]
        / df_engineered["LivingArea"].replace(0, np.nan)
    )

    df_engineered["building_coverage_ratio"] = (
        df_engineered["LivingArea"]
        / df_engineered[
            "LotSizeSquareFeet"
        ].replace(0, np.nan)
    )

    df_engineered["unused_lot_area"] = (
        df_engineered["LotSizeSquareFeet"]
        - df_engineered["LivingArea"]
    )

In [ ]:
# story features 

if "Stories" in df_engineered.columns:

    df_engineered["single_story_flag"] = (
        df_engineered["Stories"] == 1
    ).astype(int)

    df_engineered["multi_story_flag"] = (
        df_engineered["Stories"] > 1
    ).astype(int)


if "Stories" in df_engineered.columns:

    df_engineered["single_story_flag"] = (
        df_engineered["Stories"] == 1
    ).astype(int)

    df_engineered["multi_story_flag"] = (
        df_engineered["Stories"] > 1
    ).astype(int)

In [ ]:
missing_indicator_columns = [
    "LivingArea",
    "LotSizeSquareFeet",
    "YearBuilt",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "AssociationFee",
    "Stories",
    "MainLevelBedrooms"
]

for column in missing_indicator_columns:

    if column in df_engineered.columns:

        df_engineered[
            f"{column}_missing"
        ] = (
            df_engineered[column]
            .isna()
            .astype(int)
        )

df_engineered["missing_feature_count"] = (
    df_engineered.isna().sum(axis=1)
)


# clean infinite values

df_engineered = df_engineered.replace(
    [np.inf, -np.inf],
    np.nan
)

In [ ]:
# fill with medians 

numeric_columns = (
    df_engineered
    .select_dtypes(include=np.number)
    .columns
)

df_engineered[numeric_columns] = (
    df_engineered[numeric_columns]
    .fillna(
        df_engineered[numeric_columns].median()
    )
)

In [ ]:
import geopandas as gpd


folder = Path("../output_csv")


school_district_file = (
    folder / "California_School_District_Areas_2024-25.geojson"
)

school_districts = gpd.read_file(
    school_district_file
)

print(school_districts.shape)
print(school_districts.crs)
print(school_districts.columns.tolist())


[
    column
    for column in school_districts.columns
    if (
        "name" in column.lower()
        or "district" in column.lower()
        or "type" in column.lower()
    )
]

district_name_column = "DistrictName"
district_type_column = "DistrictType"


df_engineered["_row_id"] = np.arange(
    len(df_engineered)
)

valid_coordinate_mask = (
    df_engineered["Latitude"].notna()
    & df_engineered["Longitude"].notna()
)

properties_with_coordinates = (
    df_engineered[
        valid_coordinate_mask
    ]
    .copy()
)

properties_without_coordinates = (
    df_engineered[
        ~valid_coordinate_mask
    ]
    .copy()
)


properties_gdf = gpd.GeoDataFrame(
    properties_with_coordinates,
    geometry=gpd.points_from_xy(
        properties_with_coordinates["Longitude"],
        properties_with_coordinates["Latitude"]
    ),
    crs="EPSG:4326"
)

properties_gdf = properties_gdf.to_crs(
    school_districts.crs
)

district_layer = school_districts[
    [
        district_name_column,
        district_type_column,
        "geometry"
    ]
].copy()

properties_joined = gpd.sjoin(
    properties_gdf,
    district_layer,
    how="left",
    predicate="within"
)

properties_joined = properties_joined.drop(
    columns=[
        "geometry",
        "index_right"
    ],
    errors="ignore"
)

properties_joined = properties_joined.drop(
    columns=[
        "geometry",
        "index_right"
    ],
    errors="ignore"
)

properties_without_coordinates[
    district_name_column
] = np.nan

properties_without_coordinates[
    district_type_column
] = np.nan

df_engineered = pd.concat(
    [
        properties_joined,
        properties_without_coordinates
    ],
    ignore_index=True
)

df_engineered = pd.concat(
    [
        properties_joined,
        properties_without_coordinates
    ],
    ignore_index=True
)

df_engineered[
    "school_district_missing"
] = (
    df_engineered[
        district_name_column
    ]
    .isna()
    .astype(int)
)

school_district_match_rate = (
    df_engineered[
        district_name_column
    ]
    .notna()
    .mean()
)

print(
    f"School district match rate: "
    f"{school_district_match_rate:.2%}"
)

(937, 54)
EPSG:4326
['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictName', 'DistrictType', 'GradeLow', 'GradeHigh', 'GradeLowCensus', 'GradeHighCensus', 'AssistStatus', 'CongressUS', 'SenateCA', 'AssemblyCA', 'UpdateNotes', 'EnrollTotal', 'EnrollCharter', 'EnrollNonCharter', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAreaSqMi', 'Shape__Area', 'Shape__Length', 'geometry']
School district match rate: 0.00%


In [ ]:
# numeric for district 
district_dummies = pd.get_dummies(
    df_engineered[
        [
            district_name_column,
            district_type_column
        ]
    ],
    prefix=[
        "school_district",
        "school_district_type"
    ],
    drop_first=True,
    dtype=int
)

df_engineered = pd.concat(
    [
        df_engineered.drop(
            columns=[
                district_name_column,
                district_type_column
            ]
        ),
        district_dummies
    ],
    axis=1
)

non_numeric_columns = (
    df_engineered
    .select_dtypes(
        exclude=np.number
    )
    .columns
    .tolist()
)

print("Non-numeric columns:", non_numeric_columns)

protected_columns = [
    target,
    "source_month"
]

remaining_text_columns = [
    column
    for column in non_numeric_columns
    if column not in protected_columns
]

df_engineered = df_engineered.drop(
    columns=remaining_text_columns,
    errors="ignore"
)

Non-numeric columns: ['source_month', 'City_Corona', 'City_Escondido', 'City_Fontana', 'City_Hemet', 'City_Hesperia', 'City_Huntington Beach', 'City_Indio', 'City_La Quinta', 'City_Lancaster', 'City_Long Beach', 'City_Los Angeles', 'City_Menifee', 'City_Moreno Valley', 'City_Murrieta', 'City_Oakland', 'City_Oceanside', 'City_Other', 'City_Palm Desert', 'City_Palmdale', 'City_Riverside', 'City_San Bernardino', 'City_San Diego', 'City_San Jose', 'City_Temecula', 'City_Victorville', 'CountyOrParish_Butte', 'CountyOrParish_Contra Costa', 'CountyOrParish_Fresno', 'CountyOrParish_Kern', 'CountyOrParish_Lake', 'CountyOrParish_Los Angeles', 'CountyOrParish_Madera', 'CountyOrParish_Merced', 'CountyOrParish_Monterey', 'CountyOrParish_Orange', 'CountyOrParish_Other', 'CountyOrParish_Riverside', 'CountyOrParish_Sacramento', 'CountyOrParish_San Benito', 'CountyOrParish_San Bernardino', 'CountyOrParish_San Diego', 'CountyOrParish_San Joaquin', 'CountyOrParish_San Luis Obispo', 'CountyOrParish_San Ma

In [ ]:
# new vs old feature count

drop_cols = [
    target,
    "source_month"
]

old_feature_cols = [
    column
    for column in df.columns
    if column not in drop_cols
]

new_feature_cols = [
    column
    for column in df_engineered.columns
    if column not in drop_cols
]

print(
    "Old feature count:",
    len(old_feature_cols)
)

print(
    "New feature count:",
    len(new_feature_cols)
)

Old feature count: 200
New feature count: 73


In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
def evaluate_models(
    data,
    feature_cols,
    feature_set_name
):

    model_results = []

    months = sorted(
        data["source_month"].unique()
    )

    for X_window in [3, 6, 9, 12]:

        train_months = months[
            -(X_window + 1):-1
        ]

        test_month = months[-1]

        train_df = data[
            data["source_month"].isin(
                train_months
            )
        ].copy()

        test_df = data[
            data["source_month"]
            == test_month
        ].copy()

        X_train = train_df[feature_cols]
        y_train = train_df[target]

        X_test = test_df[feature_cols]
        y_test = test_df[target]

        for model_name, model in models.items():

            model.fit(
                X_train,
                y_train
            )

            train_preds = model.predict(
                X_train
            )

            test_preds = model.predict(
                X_test
            )

            train_r2 = r2_score(
                y_train,
                train_preds
            )

            test_r2 = r2_score(
                y_test,
                test_preds
            )

            mae = mean_absolute_error(
                y_test,
                test_preds
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_test,
                    test_preds
                )
            )

            model_results.append({
                "feature_set": feature_set_name,
                "model": model_name,
                "training_window_months": X_window,
                "train_months": ", ".join(
                    train_months
                ),
                "test_month": test_month,
                "feature_count": len(
                    feature_cols
                ),
                "train_r2": train_r2,
                "test_r2": test_r2,
                "mae": mae,
                "rmse": rmse
            })

    return pd.DataFrame(
        model_results
    )

In [ ]:
old_results = evaluate_models(
    data=df,
    feature_cols=old_feature_cols,
    feature_set_name="Old feature set"
)

old_results.sort_values(
    by="test_r2",
    ascending=False
)

In [ ]:
new_results = evaluate_models(
    data=df_engineered,
    feature_cols=new_feature_cols,
    feature_set_name=(
        "Engineered features "
        "+ school district"
    )
)

new_results.sort_values(
    by="test_r2",
    ascending=False
)

,feature_set,model,training_window_months,train_months,test_month,feature_count,train_r2,test_r2,mae,rmse
9,Engineered features + school district,Linear Regression,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,73,0.018076,0.326221,5.652492e+05,1.378011e+06
0,Engineered features + school district,Linear Regression,3,"202602, 202603, 202604",202605,73,0.013349,0.279832,6.250896e+05,1.424659e+06
6,Engineered features + school district,Linear Regression,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,73,0.012915,0.231245,7.584158e+05,1.471932e+06
3,Engineered features + school district,Linear Regression,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,73,0.017129,0.122270,8.480873e+05,1.572803e+06
8,Engineered features + school district,Random Forest,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,73,0.842249,-2.738423,5.685431e+05,3.245922e+06
11,Engineered features + school district,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,73,0.846840,-3.567665,5.629988e+05,3.587906e+06
5,Engineered features + school district,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,73,0.820231,-6.862429,5.276454e+05,4.707304e+06
2,Engineered features + school district,Random Forest,3,"202602, 202603, 202604",202605,73,0.868878,-9.371375,5.705269e+05,5.406448e+06
1,Engineered features + school district,Decision Tree,3,"202602, 202603, 202604",202605,73,1.000000,-30.951011,6.251172e+05,9.489344e+06
4,Engineered features + school district,Decision Tree,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,73,1.000000,-41.845281,5.562350e+05,1.098868e+07


In [ ]:
feature_comparison_df = pd.concat(
    [
        old_results,
        new_results
    ],
    ignore_index=True
)

feature_comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,feature_set,model,training_window_months,train_months,test_month,feature_count,train_r2,test_r2,mae,rmse
21,Engineered features + school district,Linear Regression,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,73,0.018076,3.262208e-01,5.652492e+05,1.378011e+06
12,Engineered features + school district,Linear Regression,3,"202602, 202603, 202604",202605,73,0.013349,2.798316e-01,6.250896e+05,1.424659e+06
18,Engineered features + school district,Linear Regression,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,73,0.012915,2.312452e-01,7.584158e+05,1.471932e+06
15,Engineered features + school district,Linear Regression,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,73,0.017129,1.222698e-01,8.480873e+05,1.572803e+06
20,Engineered features + school district,Random Forest,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,73,0.842249,-2.738423e+00,5.685431e+05,3.245922e+06
5,Old feature set,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,200,0.826019,-3.393380e+00,3.272215e+05,3.518789e+06
23,Engineered features + school district,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,73,0.846840,-3.567665e+00,5.629988e+05,3.587906e+06
17,Engineered features + school district,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,73,0.820231,-6.862429e+00,5.276454e+05,4.707304e+06
11,Old feature set,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,200,0.844988,-8.929586e+00,3.549240e+05,5.290046e+06
2,Old feature set,Random Forest,3,"202602, 202603, 202604",202605,200,0.874255,-9.261289e+00,4.264415e+05,5.377679e+06


In [ ]:
feature_comparison_df = pd.concat(
    [
        old_results,
        new_results
    ],
    ignore_index=True
)

feature_comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,feature_set,model,training_window_months,train_months,test_month,feature_count,train_r2,test_r2,mae,rmse
21,Engineered features + school district,Linear Regression,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,73,0.018076,3.262208e-01,5.652492e+05,1.378011e+06
12,Engineered features + school district,Linear Regression,3,"202602, 202603, 202604",202605,73,0.013349,2.798316e-01,6.250896e+05,1.424659e+06
18,Engineered features + school district,Linear Regression,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,73,0.012915,2.312452e-01,7.584158e+05,1.471932e+06
15,Engineered features + school district,Linear Regression,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,73,0.017129,1.222698e-01,8.480873e+05,1.572803e+06
20,Engineered features + school district,Random Forest,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,73,0.842249,-2.738423e+00,5.685431e+05,3.245922e+06
5,Old feature set,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,200,0.826019,-3.393380e+00,3.272215e+05,3.518789e+06
23,Engineered features + school district,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,73,0.846840,-3.567665e+00,5.629988e+05,3.587906e+06
17,Engineered features + school district,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,73,0.820231,-6.862429e+00,5.276454e+05,4.707304e+06
11,Old feature set,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,200,0.844988,-8.929586e+00,3.549240e+05,5.290046e+06
2,Old feature set,Random Forest,3,"202602, 202603, 202604",202605,200,0.874255,-9.261289e+00,4.264415e+05,5.377679e+06


In [ ]:
r2_comparison_table = (
    feature_comparison_df
    .pivot_table(
        index=[
            "model",
            "training_window_months"
        ],
        columns="feature_set",
        values="test_r2"
    )
    .reset_index()
)

r2_comparison_table.columns.name = None

r2_comparison_table

,model,training_window_months,Engineered features + school district,Old feature set
0,Decision Tree,3,-30.951011,-2.834751e+01
1,Decision Tree,6,-41.845281,-1.525782e+01
2,Decision Tree,9,-47.236794,-9.048902e+01
3,Decision Tree,12,-218.600558,-2.047580e+01
4,Linear Regression,3,0.279832,-1.208324e+18
5,Linear Regression,6,0.122270,-1.497904e+18
6,Linear Regression,9,0.231245,-1.568102e+16
7,Linear Regression,12,0.326221,-7.287605e+13
8,Random Forest,3,-9.371375,-9.261289e+00
9,Random Forest,6,-6.862429,-3.393380e+00


In [ ]:
r2_comparison_table = (
    feature_comparison_df
    .pivot_table(
        index=[
            "model",
            "training_window_months"
        ],
        columns="feature_set",
        values="test_r2"
    )
    .reset_index()
)

r2_comparison_table.columns.name = None

r2_comparison_table

,model,training_window_months,Engineered features + school district,Old feature set
0,Decision Tree,3,-30.951011,-2.834751e+01
1,Decision Tree,6,-41.845281,-1.525782e+01
2,Decision Tree,9,-47.236794,-9.048902e+01
3,Decision Tree,12,-218.600558,-2.047580e+01
4,Linear Regression,3,0.279832,-1.208324e+18
5,Linear Regression,6,0.122270,-1.497904e+18
6,Linear Regression,9,0.231245,-1.568102e+16
7,Linear Regression,12,0.326221,-7.287605e+13
8,Random Forest,3,-9.371375,-9.261289e+00
9,Random Forest,6,-6.862429,-3.393380e+00


In [ ]:
best_baseline_row = (
    old_results[
        old_results["model"]
        == "Linear Regression"
    ]
    .sort_values(
        by="test_r2",
        ascending=False
    )
    .iloc[0]
)

best_baseline_r2 = (
    best_baseline_row["test_r2"]
)

best_baseline_window = (
    best_baseline_row[
        "training_window_months"
    ]
)

print(
    "Best baseline window:",
    best_baseline_window
)

print(
    f"Best baseline test R²: "
    f"{best_baseline_r2:.4f}"
)

Best baseline window: 12
Best baseline test R²: -72876046266610.8594


In [ ]:
# Sort models by highest test R²
comparison_df = comparison_df.sort_values(
    by="test_r2",
    ascending=False
).reset_index(drop=True)

# Save all model comparison results
output_file = folder / "model_comparison_results.csv"

comparison_df.to_csv(
    output_file,
    index=False
)

# Select the best overall model
best_model = comparison_df.iloc[0]

print("Best Model:", best_model["model"])
print(
    "Best training window:",
    best_model["training_window_months"],
    "months"
)
print("Training months:", best_model["train_months"])
print("Test month:", best_model["test_month"])
print(f"Training R²: {best_model['train_r2']:.4f}")
print(f"Test R²: {best_model['test_r2']:.4f}")
print(f"Test MAE: ${best_model['mae']:,.2f}")
print(f"Test RMSE: ${best_model['rmse']:,.2f}")

best_model_df = comparison_df.iloc[[0]]

best_model_file = folder / "best_model_result.csv"

best_model_df.to_csv(
    best_model_file,
    index=False
)

Best Model: Random Forest
Best training window: 6 months
Training months: 202511, 202512, 202601, 202602, 202603, 202604
Test month: 202605
Training R²: 0.8260
Test R²: -3.3934
Test MAE: $327,221.51
Test RMSE: $3,518,789.41
